# 8. Ders: Kelimeleri Sayılara Dönüştürmek (Doğal Dil İşlemeye Giriş)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Önceki dersimizde makine öğrenmesi modeline insanların sıcaklık değerlerini (36.5, 38.2) verdik. Çünkü bilgisayarlar temel donanımları (CPU, GPU) gereği sadece matematikten, yani rakamlardan anlarlar.

Peki, yapay zekaya bir kitap okutup ondan bir özet çıkarmasını veya yazdığımız metinden duygu analiz etmesini istersek ne olur? Bilgisayar "Merhaba, bugün hava çok güzel" cümlesinden hiçbir şey anlamaz. Doğal Dil İşleme (NLP) biliminin ilk kuralı şudur: İnsan dilini (metinleri), makinenin anlayabileceği matematiksel parçalara (sayılara) dönüştürmeliyiz. Bu dersimizde bir cümleyi nasıl kelime kelime parçalayıp (Tokenization) sayılabilir verilere çevirdiğimizi öğreneceğiz.

## Adım 2: Teknik Terimler ve Sözlük

- NLP (Natural Language Processing): Doğal Dil İşleme. Bilgisayarların insan dilini anlamasını sağlayan bilim dalı.
- Token: Bir metnin parçalanmış en küçük yapı taşıdır (genellikle kelimeler veya noktalama işaretleri).
- Tokenization (Parçalama/Jetonlama): Bütün bir cümleyi veya paragrafı tokenlarına ayırma işlemidir.
- Stop Words (Etkisiz Kelimeler): Cümlede dil bilgisi amacıyla bulunan ama anlama tek başına pek bir şey katmayan, yapay zekayı gereksiz yoran kelimelerdir (Örn: ve, veya, ile, ama).
- NLTK (Natural Language Toolkit): Python'da metin parçalama ve dil bilgisi analizleri için kullanılan en meşhur açık kaynaklı klasik kütüphanedir.

## Adım 3: Temel String Parçalama (.split metodu)

Python'da kelimeleri boşluklardan bölmek için basit bir string metodu vardır. Önce bunu deneyelim.

In [ ]:
cumle = "Yapay Zeka öğrenmek çok eğlenceli ve heyecan verici!"

# .split() komutu varsayılan olarak cümleyi boşluklardan (space) keserek bir Liste oluşturur.
basit_parcalar = cumle.split()

print("Bölünmüş Liste:")
print(basit_parcalar)

print(f"\nToplam kelime sayısı: {len(basit_parcalar)}")

# Ancak dikkat ederseniz 'verici!' kelimesinin sonundaki ünlem işareti kelimeye yapışık kaldı. 
# Bilgisayar için 'verici' ile 'verici!' tamamen farklı iki sayıya denk gelir. Bu bir sorundur.

## Adım 4: NLTK Kütüphanesini Kurma ve İçe Aktarma

Basit `.split()` metodunun eksiklerini gidermek için NLTK (Natural Language Toolkit) kütüphanesini kullanacağız.

In [ ]:
# Colab üzerinde NLTK yüklü gelir, ancak kullanacağımız noktalama paketi (punkt) için özel indirme yapmalıyız.
import nltk

# İngilizce ve genel kelime ayırıcı zeka paketini koda indiriyoruz.
nltk.download('punkt')

print("\nNLTK kütüphanesi ve 'punkt' paketi başarıyla yüklendi.")

## Adım 5: Cümleleri Akıllı Parçalama (word_tokenize)

Şimdi aynı cümleyi NLTK'nın yapay zekası ile parçalayalım.

In [ ]:
from nltk.tokenize import word_tokenize

cumle = "Yapay Zeka öğrenmek çok eğlenceli ve heyecan verici!"

akilli_parcalar = word_tokenize(cumle)

print("Akıllı Tokenization Sonucu:")
print(akilli_parcalar)

# Çıktıya bakarsanız 'verici' ile ünlem '!' işaretinin birbirinden akıllıca ayrıldığını görürsünüz. 
# Her ikisi de artık ayrı birer tokendir.

## Adım 6: Noktalama İşaretlerini Temizleme

Bir metnin duygusunu veya özetini analiz ederken virgüller ve noktalar genellikle işimize yaramaz (bazı modern modeller hariç). Python'daki `.isalnum()` (is alpha numeric) metodunu kullanarak sadece harflerden ve sayılardan oluşan tokenları filtreleyebiliriz.

In [ ]:
temiz_kelimeler = []

for token in akilli_parcalar:
    # Eğer token harf veya rakam ise (noktalama işareti değilse)
    if token.isalnum():
        temiz_kelimeler.append(token)

print("Noktalamalardan Arındırılmış Kelimeler:")
print(temiz_kelimeler)

## Adım 7: Kelime Frekansını (Sıklığını) Sayma

Metni parçaladıktan sonra yapay zekanın ilk baktığı şey, hangi kelimeden kaç tane olduğudur. Bu metnin içeriği hakkında en büyük ipucunu verir.

In [ ]:
from nltk.probability import FreqDist

uzun_metin = "yapay zeka öğrenmek güzel. yapay zeka ile projeler yapmak daha da güzel. zeka her şeydir."

# Hepsini küçük harfe çevirelim ki 'Yapay' ile 'yapay' aynı sayılsın
uzun_metin_kucuk = uzun_metin.lower()

# Önce parçalayalım (tokenization)
tokenlar = word_tokenize(uzun_metin_kucuk)

# Noktalamaları atalım (List Comprehension adı verilen kısa for döngüsü yazımı)
sadece_kelimeler = [kelime for kelime in tokenlar if kelime.isalnum()]

# Frekans Dağılımını Hesapla
frekans = FreqDist(sadece_kelimeler)

# En çok geçen 3 kelimeyi ekrana yazdır
print("En sık geçen kelimeler ve sayıları:")
print(frekans.most_common(3))

## Adım 8: Kelimeleri Sayılara Çevirme Mantığı

Son adımda, bilgisayarın anlayacağı formata (Index) dönüştürme işlemi yapılır. Her benzersiz kelimeye bir kimlik numarası (ID) atanır.

In [ ]:
# Elimizdeki benzersiz kelimelerden bir kimlik sözlüğü oluşturuyoruz.
kelime_sozlugu = {
    "yapay": 1,
    "zeka": 2,
    "öğrenmek": 3,
    "güzel": 4,
    "ile": 5
}

ornek_dizi = ["yapay", "zeka", "güzel"]
matematiksel_vektor = []

for kelime in ornek_dizi:
    # Sözlükteki rakamsal karşılığını bul ve listeye ekle
    sayisal_deger = kelime_sozlugu[kelime]
    matematiksel_vektor.append(sayisal_deger)

print("İnsanların gördüğü cümle:", ornek_dizi)
print("Yapay zekanın gördüğü matris:", matematiksel_vektor)
# İşte ChatGPT gibi modeller de tam olarak sizin kelimelerinizi bu şekilde ID numaralarına çevirip işler!

## Adım 9: Dene ve Keşfet - Pandas ile Kelime Tablosu

Önceki derslerde öğrendiğimiz Pandas'ı kullanarak kelime sayımlarını (frekanslarını) daha şık bir tablo haline getirelim.

In [ ]:
import pandas as pd

# FreqDist'in oluşturduğu yapıyı doğrudan DataFrame'e çevirebiliriz
kelime_sayilari = frekans.items()

df_frekans = pd.DataFrame(kelime_sayilari, columns=["Kelime", "Geçiş Sayısı"])

# En çok geçene göre (Geçiş Sayısı) büyükten küçüğe sıralayalım
df_frekans = df_frekans.sort_values(by="Geçiş Sayısı", ascending=False)

print("Kelime Sıklığı Tablosu:")
print(df_frekans)

## Adım 10: Sıra Sende Görevi - Stop Words Temizleyici

Problem: Yapay zekanın cümleyi anlamasını sağlarken "ve, ile, bu" gibi kelimeler gereksiz yere yer kaplar. Sizden bir metni önce parçalayıp (tokenize), sonra gereksiz kelimeleri çöpe atan bir filtre yazmanız isteniyor.

Gereksinimler:
1. `ham_metin = "Python ve yapay zeka ile çok güzel uygulamalar yazabiliriz ama çalışmak şart."` değişkenini oluşturun.
2. Metni `word_tokenize()` ile parçalayarak bir listeye alın.
3. `yasakli_kelimeler = ["ve", "ile", "çok", "ama"]` adında bir liste (stop words) tanımlayın.
4. Bir for döngüsü kurun. Sadece `yasakli_kelimeler` listesinde OLMAYAN tokenları `temiz_metin` adlı yeni bir listeye ekleyin (İpucu: `if kelime not in yasakli_kelimeler:` yapısını kullanın).
5. Sonuçta kalan değerli kelimeleri ekrana yazdırın.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 8. Dersi tamamlayarak Doğal Dil İşlemenin (NLP) kalbi olan Tokenization sürecini çözdünüz. Artık metinleri makinenin anlayabileceği sayılara ve yığınlara nasıl çevireceğinizi biliyorsunuz. 

Bir sonraki derste, dünyanın en büyük yapay zeka kütüphanesi Hugging Face'e giriş yapacak ve devasa şirketler tarafından eğitilmiş modelleri kodlarımıza entegre ederek MoodJar (His Kavanozu) projesini hayata geçireceğiz.